In [1]:
%cd /content
!rm -rf RiskAware-Complaints-Engine
!git clone https://github.com/Nusultan11/RiskAware-Complaints-Engine.git
%cd /content/RiskAware-Complaints-Engine
!pip install -e .

/content
Cloning into 'RiskAware-Complaints-Engine'...
remote: Enumerating objects: 266, done.
remote: Counting objects: 100% (266/266), done.
remote: Compressing objects: 100% (178/178), done.
remote: Total 266 (delta 106), reused 225 (delta 66), pack-reused 0 (from 0)
Receiving objects: 100% (266/266), 165.39 KiB | 6.62 MiB/s, done.
Resolving deltas: 100% (106/106), done.
/content/RiskAware-Complaints-Engine
Obtaining file:///content/RiskAware-Complaints-Engine
  Installing build dependencies ... done
  Checking if build backend supports build_editable ... done
  Getting requirements to build editable ... done
  Preparing editable metadata (pyproject.toml) ... done
  Building editable for risk-aware-complaints-engine (pyproject.toml) ... done
  Created wheel for risk-aware-complaints-engine: filename=risk_aware_complaints_engine-0.1.0-0.editable-py3-none-any.whl size=2362 sha256=6a55c52939e779ce8e0ab6592366e663ab0cab067f9ddb38f0fb81fc2be497ac
  Stored in directory: /tmp/pip-ephem-whe

In [13]:
import os
import sys
import json
import glob
import shutil
import zipfile
from pathlib import Path

import numpy as np
import pandas as pd
import yaml
import torch
import matplotlib.pyplot as plt
from torch.utils.data import TensorDataset, DataLoader

from sklearn.metrics import accuracy_score, f1_score, classification_report
from google.colab import drive, files

sys.path.insert(0, os.path.abspath("src"))

from risk_aware.config import load_project_configs
from risk_aware.preprocessing.neural import (
    neural_clean,
    simple_tokenize,
    NeuralTextPreprocessor,
)
from scripts.train_lstm_category import BiLSTMClassifier, _load_npz

In [3]:
print("cuda_available:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("gpu:", torch.cuda.get_device_name(0))

cuda_available: True
gpu: Tesla T4


In [4]:
drive.mount('/content/drive')

ROOT = "/content/RiskAware-Complaints-Engine"
os.makedirs(f"{ROOT}/data/raw", exist_ok=True)

cands = glob.glob("/content/drive/MyDrive/**/cfpb_complaints.csv", recursive=True)
assert cands, "cfpb_complaints.csv not found in Google Drive"
src = cands[0]
dst = f"{ROOT}/data/raw/cfpb_complaints.csv"
shutil.copy(src, dst)

print("copied:", src, "->", dst)
print("exists:", os.path.exists(dst))

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
copied: /content/drive/MyDrive/RiskAware Complaints Engine/data/raw/cfpb_complaints.csv -> /content/RiskAware-Complaints-Engine/data/raw/cfpb_complaints.csv
exists: True


In [5]:
out_dir = Path("reports/metrics")
out_dir.mkdir(parents=True, exist_ok=True)

In [6]:
cfg_path = Path("/content/RiskAware-Complaints-Engine/configs/category.yaml")
cfg = yaml.safe_load(cfg_path.read_text())

b = cfg["stacks"]["bilstm"]
b["architecture"] = "bilstm_only"
b["max_vocab_size"] = 50000
b["min_token_freq"] = 2
b["max_length"] = 384

b["embedding_dim"] = 128
b["lstm_hidden_dim"] = 128
b["bilstm_hidden_dim"] = 128
b["num_layers_lstm"] = 1
b["num_layers_bilstm"] = 1
b["dropout"] = 0.3
b["epochs"] = 12
b["batch_size"] = 64
b["learning_rate"] = 0.001
b["weight_decay"] = 1e-5
b["grad_clip_norm"] = 1.0
b["early_stopping_patience"] = 3
b["use_class_weights"] = True
b["class_weight_max"] = 3.0

cfg_path.write_text(yaml.safe_dump(cfg, sort_keys=False))
print("Updated bilstm config:")
print(b)

Updated bilstm config:
{'architecture': 'bilstm_only', 'max_vocab_size': 50000, 'min_token_freq': 2, 'max_length': 384, 'embedding_dim': 128, 'lstm_hidden_dim': 128, 'bilstm_hidden_dim': 128, 'num_layers_lstm': 1, 'num_layers_bilstm': 1, 'dropout': 0.3, 'epochs': 12, 'batch_size': 64, 'learning_rate': 0.001, 'weight_decay': 1e-05, 'grad_clip_norm': 1.0, 'early_stopping_patience': 3, 'use_class_weights': True, 'class_weight_max': 3.0}


In [7]:
samples = [
    "I don't think this is correct",
    "They can't verify the payment",
    "We've been charged twice",
]

for s in samples:
    print("RAW   :", s)
    print("CLEAN :", neural_clean(s))
    print("TOKENS:", simple_tokenize(s))
    print("---")

RAW   : I don't think this is correct
CLEAN : i don't think this is correct
TOKENS: ['i', "don't", 'think', 'this', 'is', 'correct']
---
RAW   : They can't verify the payment
CLEAN : they can't verify the payment
TOKENS: ['they', "can't", 'verify', 'the', 'payment']
---
RAW   : We've been charged twice
CLEAN : we've been charged twice
TOKENS: ["we've", 'been', 'charged', 'twice']
---


In [8]:
!PYTHONPATH=src python -u src/risk_aware/data/prepare.py
!PYTHONPATH=src python -u scripts/prepare_lstm_data.py

/usr/local/lib/python3.12/dist-packages/sklearn/model_selection/_split.py:1023: UserWarning: The least populated class in y has only 3 members, which is less than n_splits=5.
  warnings.warn(
LSTM preprocessing completed.
Saved tensors to: artifacts/lstm_preprocessing


In [9]:
cfg_loaded = load_project_configs("configs")
text_col = cfg_loaded["base"]["data"]["text_column"]

train_df = pd.read_csv("data/processed/train.csv")
val_df = pd.read_csv("data/processed/val.csv")
test_df = pd.read_csv("data/processed/test.csv")

x_train = train_df[text_col].astype(str).tolist()
x_val = val_df[text_col].astype(str).tolist()
x_test = test_df[text_col].astype(str).tolist()

prep = NeuralTextPreprocessor(max_vocab_size=50000, min_token_freq=2, max_length=384)
prep.fit(x_train)
vocab = prep.vocab
assert vocab is not None
token_to_id = vocab.token_to_id

def oov_stats(texts):
    total_tokens, unk_tokens = 0, 0
    for t in texts:
        toks = simple_tokenize(str(t))
        total_tokens += len(toks)
        unk_tokens += sum(1 for tok in toks if tok not in token_to_id)
    covered_tokens = total_tokens - unk_tokens
    return {
        "total_tokens": int(total_tokens),
        "covered_tokens": int(covered_tokens),
        "unk_tokens": int(unk_tokens),
        "oov_rate": float(unk_tokens / total_tokens if total_tokens else 0.0),
    }

train_oov = oov_stats(x_train)
val_oov = oov_stats(x_val)
test_oov = oov_stats(x_test)

print("train:", train_oov)
print("val  :", val_oov)
print("test :", test_oov)

Path("reports/metrics/lstm_oov_stats_maxlen_384.json").write_text(
    json.dumps({"train": train_oov, "val": val_oov, "test": test_oov}, indent=2),
    encoding="utf-8",
)

train: {'total_tokens': 8135679, 'covered_tokens': 8115173, 'unk_tokens': 20506, 'oov_rate': 0.0025205025911174714}
val  : {'total_tokens': 2046073, 'covered_tokens': 2037981, 'unk_tokens': 8092, 'oov_rate': 0.003954893104986968}
test : {'total_tokens': 2556809, 'covered_tokens': 2546562, 'unk_tokens': 10247, 'oov_rate': 0.004007729947759101}


419

In [10]:
!PYTHONPATH=src python -u scripts/train_lstm_category.py 2>&1 | tee train_lstm.log

device=cuda architecture=bilstm_only use_class_weights=True learning_rate=0.001
epoch=1 train_loss=4.3524 val_macro_f1=0.0415 val_accuracy=0.1720 lr=0.001000
epoch=2 train_loss=3.5161 val_macro_f1=0.0788 val_accuracy=0.1924 lr=0.001000
epoch=3 train_loss=2.9020 val_macro_f1=0.0996 val_accuracy=0.2230 lr=0.001000
epoch=4 train_loss=2.3655 val_macro_f1=0.1182 val_accuracy=0.2529 lr=0.001000
epoch=5 train_loss=1.9482 val_macro_f1=0.1418 val_accuracy=0.2918 lr=0.001000
epoch=6 train_loss=1.5810 val_macro_f1=0.1461 val_accuracy=0.2856 lr=0.001000
epoch=7 train_loss=1.3190 val_macro_f1=0.1619 val_accuracy=0.3200 lr=0.001000
epoch=8 train_loss=1.1278 val_macro_f1=0.1680 val_accuracy=0.3223 lr=0.001000
epoch=9 train_loss=0.9504 val_macro_f1=0.1716 val_accuracy=0.3417 lr=0.001000
epoch=10 train_loss=0.8219 val_macro_f1=0.1714 val_accuracy=0.3649 lr=0.001000
epoch=11 train_loss=0.7007 val_macro_f1=0.1750 val_accuracy=0.3529 lr=0.001000
epoch=12 train_loss=0.6039 val_macro_f1=0.1699 val_accuracy=

In [11]:
!cat reports/metrics/category_lstm_metrics_val.json
!cat reports/metrics/category_lstm_metrics_test.json

{
  "macro_f1": 0.17497106433395201,
  "accuracy": 0.3528527117210264
}{
  "macro_f1": 0.17420333365329163,
  "accuracy": 0.3579183826282291
}

In [14]:
# 1) load artifacts
meta = json.loads(Path("artifacts/category_lstm/training_metadata.json").read_text())
ckpt = torch.load("artifacts/category_lstm/model.pt", map_location="cpu")

labels = ckpt["labels"]
pad_idx = ckpt["pad_idx"]
cfg = meta["config"]

# 2) rebuild model
model = BiLSTMClassifier(
    architecture=cfg["architecture"],
    vocab_size=cfg["vocab_size"],
    embedding_dim=cfg["embedding_dim"],
    lstm_hidden_dim=cfg["lstm_hidden_dim"],
    bilstm_hidden_dim=cfg["bilstm_hidden_dim"],
    num_layers_lstm=cfg["num_layers_lstm"],
    num_layers_bilstm=cfg["num_layers_bilstm"],
    n_labels=cfg["n_labels"],
    pad_idx=pad_idx,
    dropout=cfg["dropout"],
)
model.load_state_dict(ckpt["state_dict"])
model.eval()

# 3) load test tensors
input_ids, attention_mask, y_true_idx = _load_npz(Path("artifacts/lstm_preprocessing/test.npz"))

# 4) batched inference (safe)
batch_size = 128
ds = TensorDataset(input_ids, attention_mask, y_true_idx)
dl = DataLoader(ds, batch_size=batch_size, shuffle=False, num_workers=0)

pred_idx_parts = []
true_idx_parts = []

with torch.no_grad():
    for x_ids, x_mask, y in dl:
        logits = model(x_ids, x_mask)
        pred = torch.argmax(logits, dim=1)
        pred_idx_parts.append(pred.cpu().numpy())
        true_idx_parts.append(y.cpu().numpy())

y_pred_idx = np.concatenate(pred_idx_parts)
y_true_idx = np.concatenate(true_idx_parts)

# 5) decode labels
y_true = np.array([labels[int(i)] for i in y_true_idx], dtype=object)
y_pred = np.array([labels[int(i)] for i in y_pred_idx], dtype=object)

# 6) metrics
test_accuracy = float(accuracy_score(y_true, y_pred))
test_macro_f1 = float(f1_score(y_true, y_pred, average="macro"))
test_weighted_f1 = float(f1_score(y_true, y_pred, average="weighted"))

print("best_checkpoint_metric = val_macro_f1")
print("best_epoch =", meta["best_epoch"])
print("val =", json.loads(Path("reports/metrics/category_lstm_metrics_val.json").read_text()))
print("test_accuracy =", test_accuracy)
print("test_macro_f1 =", test_macro_f1)
print("test_weighted_f1 =", test_weighted_f1)

report = classification_report(y_true, y_pred, digits=4)
print("\nclassification_report(test):\n")
print(report)

Path("reports/metrics/category_lstm_classification_report_test.txt").write_text(report, encoding="utf-8")
print("saved: reports/metrics/category_lstm_classification_report_test.txt")

best_checkpoint_metric = val_macro_f1
best_epoch = 11
val = {'macro_f1': 0.17497106433395201, 'accuracy': 0.3528527117210264}
test_accuracy = 0.3579183826282291
test_macro_f1 = 0.17420333365329163
test_weighted_f1 = 0.378239460960994

classification_report(test):

                                                                  precision    recall  f1-score   support

 Bank account or service|Account opening, closing, or management     0.4701    0.3697    0.4139       468
                Bank account or service|Deposits and withdrawals     0.4124    0.2655    0.3230       275
Bank account or service|Making/receiving payments, sending money     0.1059    0.1607    0.1277       112
   Bank account or service|Problems caused by my funds being low     0.3671    0.6591    0.4715       176
               Bank account or service|Using a debit or ATM card     0.4262    0.4370    0.4315       119
                         Consumer Loan|Account terms and changes     0.0000    0.0000    0.0000   

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Recall is ill-defined and being set to 0.0 in labels with no true samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_

In [15]:
summary = {
    "best_checkpoint_metric": "val_macro_f1",
    "best_epoch": 11,
    "val": {
        "accuracy": 0.3528527117210264,
        "macro_f1": 0.17497106433395201
    },
    "test": {
        "accuracy": 0.3579183826282291,
        "macro_f1": 0.17420333365329163,
        "weighted_f1": 0.378239460960994
    },
    "notes": [
        "long-tail classes produce many zero-prediction rows in classification report",
        "warnings about undefined precision/recall are expected for rare classes"
    ]
}

Path("reports/metrics").mkdir(parents=True, exist_ok=True)
Path("reports/metrics/category_lstm_summary.json").write_text(
    json.dumps(summary, indent=2, ensure_ascii=False),
    encoding="utf-8",
)
print("saved: reports/metrics/category_lstm_summary.json")

saved: reports/metrics/category_lstm_summary.json


In [16]:
bundle_name = "lstm_baseline_final_bundle.zip"
paths_to_pack = [
    "artifacts/category_lstm",
    "artifacts/lstm_preprocessing/metadata.json",
    "artifacts/lstm_preprocessing/vocab.json",
    "reports/metrics/category_lstm_metrics_val.json",
    "reports/metrics/category_lstm_metrics_test.json",
    "reports/metrics/category_lstm_classification_report_test.txt",
    "reports/metrics/category_lstm_summary.json",
    "reports/metrics/lstm_oov_stats_maxlen_384.json",
    "configs/category.yaml",
    "train_lstm.log",
]

root = Path("/content/RiskAware-Complaints-Engine")
bundle_path = root / bundle_name

with zipfile.ZipFile(bundle_path, "w", zipfile.ZIP_DEFLATED) as z:
    for rel in paths_to_pack:
        p = root / rel
        if p.is_file():
            z.write(p, arcname=str(p.relative_to(root)))
        elif p.is_dir():
            for f in p.rglob("*"):
                if f.is_file():
                    z.write(f, arcname=str(f.relative_to(root)))

print("created:", bundle_path)

created: /content/RiskAware-Complaints-Engine/lstm_baseline_final_bundle.zip


In [17]:
files.download("/content/RiskAware-Complaints-Engine/lstm_baseline_final_bundle.zip")

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>

In [18]:
cfg_path = Path("/content/RiskAware-Complaints-Engine/configs/category.yaml")
cfg = yaml.safe_load(cfg_path.read_text())
print(cfg["stacks"].get("bert", {}))
print("next_step = distilbert baseline on same processed splits")

{'model_name': 'distilbert-base-uncased', 'max_length': 256, 'learning_rate': 2e-05, 'batch_size': 32, 'epochs': 3}
next_step = distilbert baseline on same processed splits
